# Project - Airline AI Assistant

We'll now bring together what we've learned to make an AI Customer Support assistant for an Airline

In [23]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [24]:
# Initialization

load_dotenv(override=True)

google_api_key = os.getenv('GOOGLE_API_KEY')
if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:8]}")
else:
    print("Google API Key not set")
    
MODEL = "gemini-3.5-flash-lite"
openai = OpenAI()

gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
openai = OpenAI(api_key=google_api_key, base_url=gemini_url)

# As an alternative, if you'd like to use Ollama instead of OpenAI
# Check that Ollama is running for you locally (see week1/day2 exercise) then uncomment these next 2 lines
MODEL_OLLAMA = "llama3.2"
ollama = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')

Google API Key exists and begins AQ.Ab8RN


In [25]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

## Tools

Tools are an incredibly powerful feature provided by the frontier LLMs.

With tools, you can write a function, and have the LLM call that function as part of its response.

Sounds almost spooky.. we're giving it the power to run code on our machine?

Well, kinda.

In [26]:
# There's a particular dictionary structure that's required to describe our function:

get_price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

set_price_function = {
    "name": "set_ticket_price",
    "description": "Set the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
            "price": {
              "type": "number",
              "description": "The price of the ticket"
            }
        },
        "required": ["destination_city", "price"],
        "additionalProperties": False
    }
}

In [27]:
# And this is included in a list of tools:

tools = [{"type": "function", "function": get_price_function}, {"type": "function", "function": set_price_function}]

### DB Code

In [28]:
import sqlite3

In [29]:
DB = "prices.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)')
    conn.commit()

In [30]:
def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"

In [31]:
def set_ticket_price(city, price):
    print(f"DATABASE TOOL CALLED: Setting price for {city} price for {price}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?', (city.lower(), price, price))
        conn.commit()

### Handle Tools

In [32]:
available_functions = {
  'get_ticket_price': get_ticket_price, 
  'set_ticket_price': set_ticket_price
}

In [33]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        
        function_name = tool_call.function.name
        function_to_call = available_functions.get(function_name)

        if not function_to_call:
             responses.append({
                "role": "tool",
                "content": f'Error: Function {function_name} not available',
                "tool_call_id": tool_call.id
            })

        arguments = json.loads(tool_call.function.arguments)
        city = arguments.get('destination_city')


        if tool_call.function.name == "set_ticket_price":
            price = arguments.get('price')
            price_details = function_to_call(city, price)
            responses.append({
                "role": "tool",
                "content": f"Ticket price {price} for this city {city} updated successfully",
                "tool_call_id": tool_call.id
            })

        if tool_call.function.name == "get_ticket_price":
            city = arguments.get('destination_city')
            price_details = function_to_call(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
            
    return responses

In [34]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    
    return response.choices[0].message.content

### Run Chat Interface

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7881
* To create a public link, set `share=True` in `launch()`.


DATABASE TOOL CALLED: Setting price for paris price for 1420
DATABASE TOOL CALLED: Setting price for tokiyo price for 1000
DATABASE TOOL CALLED: Setting price for india price for 600
DATABASE TOOL CALLED: Setting price for sigapore price for 700
DATABASE TOOL CALLED: Setting price for dubai price for 705
DATABASE TOOL CALLED: Setting price for kuwait price for 450
DATABASE TOOL CALLED: Getting price for paris
